# XStore Benchmarks – Big Experiments

## Libraries

In [ ]:
from fabric import ThreadingGroup, Connection
from chi import server, ssh
import os
import warnings
import time
import sys
sys.path.append('../scripts')
from utilities import *
warnings.filterwarnings('ignore')

## Configurations

In [ ]:
sshUser = "ubuntu"
server_ips = ["172.16.5.141"]
server_local_ips = {}
client_ips = ["172.16.5.134", "172.16.5.136", "172.16.5.138", "172.16.5.133",
              "172.16.5.135", "172.16.5.137", "172.16.5.139", "172.16.5.140"]

storageType = {"HDD": "/HDD", "NVME": "/NVME_RAID-0"}
SERVER_WORKING_DIR = storageType["NVME"]


## Initialize environment


In [ ]:
# These groups are for processing in parallel 
serverConnections = [makeConnection(i, user=sshUser) for i in server_ips]
clientConnections = [makeConnection(i, user=sshUser) for i in client_ips]
allConnections = (serverConnections + clientConnections)
serverGroup = ThreadingGroup.from_connections(serverConnections)
clientGroup = ThreadingGroup.from_connections(clientConnections)
allGroup = ThreadingGroup.from_connections(allConnections)

# Get local IP for server (every experiment cell needs this)
local_ips = serverGroup.run("hostname -I | cut -d ' ' -f 1", hide=True)
for i in local_ips:
    server_local_ips[i.host] = local_ips[i].stdout.strip()

# Update & Upgrade latest packages (tolerate transient apt locks: remote apt-get survives a failed cell,
# and unattended-upgrades races the first run on fresh nodes; wait on the lock instead of aborting the cell)
apt = allGroup.run("sudo apt-get update -o DPkg::Lock::Timeout=180 && sudo apt-get upgrade -y -o DPkg::Lock::Timeout=180", hide=True, warn=True)
for rconn, r in apt.items():
    if r.exited != 0:
        print(f"[apt] {rconn.host}: exit={r.exited}: {r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ''}")
    else:
        print(f"[apt] {rconn.host}: ok")
rebootGroup(allGroup)

Wait for all server to come online after reboot

In [ ]:
# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

## Datasets

Preparing test datasets on client nodes

In [ ]:
clientGroup.put("../scripts/initData.sh")
clientGroup.run("bash initData.sh MYSTIC@IIT", hide=True);

## Benchmark server (XStore)

Prepare benchmark server

In [ ]:
serverGroup.put("../scripts/_common.sh")
serverGroup.put("../scripts/XStore/initXStoreServer.sh")
serverGroup.run(f"sudo chmod -R 777 {SERVER_WORKING_DIR}; bash initXStoreServer.sh {SERVER_WORKING_DIR}", hide=True);

## Benchmark client (XBench)
Prepare benchmark client

In [ ]:
clientGroup.put("../scripts/initXBenchClient.sh")
clientGroup.run("bash initXBenchClient.sh", hide=True)
clientGroup.close()

## Experiments - Settings

In [ ]:
targetThreads = [1, 2, 4, 8, 16, 32, 64]
numClients = [i * len(client_ips) for i in targetThreads]

## Experiment – UNARY INSERT (SEQ)
**Perform experiments**

In [ ]:
runWorkload("INSERT_UNARY_SEQ", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("INSERT_UNARY_SEQ", "XStore", targetThreads, client_ips, "Insert")

## Experiment – UNARY INSERT (RAND)
**Perform experiments**

In [ ]:
runWorkload("INSERT_UNARY_RAND", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("INSERT_UNARY_RAND", "XStore", targetThreads, client_ips, "Insert")

## Experiment – BATCH INSERT (SEQ)
**Perform experiments**

In [ ]:
runWorkload("INSERT_BATCH_SEQ", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("INSERT_BATCH_SEQ", "XStore", targetThreads, client_ips)

## Experiment – BATCH INSERT (RAND)
**Perform experiments**

In [ ]:
runWorkload("INSERT_BATCH_RAND", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("INSERT_BATCH_RAND", "XStore", targetThreads, client_ips)

## Bootstrap Database

Reboot server to a fresh state to avoid becoming non-accessible from public network


In [ ]:
rebootGroup(allGroup)

# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

<div class="alert alert-block alert-info">Note: This is required so that there's actual data for clients to query from</div>

In [ ]:
serverGroup.put("../scripts/restoreDB.sh")
serverGroup.put("../scripts/_common.sh")

# Before restoring: stop any running server cleanly (SIGTERM via _common.sh, docs/XStore_Handoff.md S9) so it
# is not holding DataStore.lock, then drop the stale WAL left by the insert phase -- its replay would otherwise
# run on top of the restored DB and corrupt the benchmark data.
kill = serverGroup.run("source _common.sh && killTmuxIfRunning 5", hide=True)
for resconn, res in kill.items():
    print(f"{resconn.host}: {res.stdout.strip() or 'no tmux session'}")
serverGroup.run(f"rm -rf {SERVER_WORKING_DIR}/XStore_Server/build/DataStore/WAL", hide=True)

# Restore DB
serverGroup.run(f"bash restoreDB.sh MYSTIC@IIT XStore BENCH_DB {SERVER_WORKING_DIR}", hide=True)

# Keep checking if job is done running in tmux window
checkTmuxJobIsDone(serverGroup);

### Post-restore start check

Start the server once, print the seconds-to-port (restored sidecars should make this fast; minutes mean stale or
old-format indexes -- see docs/BENCH_DB_Index_Refresh_Runbook.md), stop it cleanly, start again: the second number
proves the sidecar set is clean and current-format.

In [ ]:
# First start after restore: seconds-to-port with the restored indexes
serverGroup.put("../scripts/_common.sh")
serverGroup.put("../scripts/XStore/runXStoreServer.sh")
first = serverGroup.run(f"bash runXStoreServer.sh false {SERVER_WORKING_DIR}", hide=True)
for resconn, res in first.items():
    print(f"{resconn.host}: {res.stdout.strip()}")

In [ ]:
# Stop cleanly (S9 helper): XStore exits on its own, indexes left clean
stopped = serverGroup.run("source _common.sh && killTmuxIfRunning 2", hide=True)
for resconn, res in stopped.items():
    print(f"{resconn.host}: {res.stdout.strip() or 'no tmux session'}")

In [ ]:
# Second start: must be seconds, not minutes
again = serverGroup.run(f"bash runXStoreServer.sh false {SERVER_WORKING_DIR}", hide=True)
for resconn, res in again.items():
    print(f"{resconn.host}: {res.stdout.strip()}")

## Experiment – UNARY QUERY (SEQ)

**Perform experiments**

In [ ]:
runWorkload("QUERY_UNARY_SEQ", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("QUERY_UNARY_SEQ", "XStore", targetThreads, client_ips, "Query", logScale=True)

## Experiment – UNARY QUERY (RAND)

**Perform experiments**

In [ ]:
runWorkload("QUERY_UNARY_RAND", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 1000, 0, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
unaryPlots("QUERY_UNARY_RAND", "XStore", targetThreads, client_ips, "Query", logScale=True)

## Experiment – BATCH QUERY (SEQ)

**Perform experiments**

In [ ]:
runWorkload("QUERY_BATCH_SEQ", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 100, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("QUERY_BATCH_SEQ", "XStore", targetThreads, client_ips, 100)

## Experiment – BATCH QUERY (RAND)

**Perform experiments**

In [ ]:
runWorkload("QUERY_BATCH_RAND", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 100, 10, serverGroup, SERVER_WORKING_DIR, "ubuntu")

**Visualization**

In [ ]:
batchPlots("QUERY_BATCH_RAND", "XStore", targetThreads, client_ips, 100)

## Refresh Database

Reboot server to a fresh state to avoid becoming non-accessible from public network


In [ ]:
rebootGroup(allGroup)

# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)

for ip in (server_ips + client_ips):
    waitForSSH(ip, user=sshUser, timeout=1200)

## Experiment – MIN AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_MIN_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_MIN_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – MAX AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_MAX_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_MAX_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – SUM AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_SUM_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_SUM_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – AVG AGG QUERY

**Perform experiments**

In [ ]:
runWorkload("AGG_AVG_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_AVG_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Extended aggregates

Same sweep shape as MIN/MAX/SUM/AVG (n=60, batchSize=1000) but the workload directories are newer:
smoke-test one thread-count step first, then run the full cell, and report these separately from the core AGG set.

## Experiment – VARIANCE AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_VARIANCE_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_VARIANCE_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – STDDEV AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_STDDEV_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_STDDEV_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – DISTINCTCOUNT AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_DISTINCTCOUNT_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_DISTINCTCOUNT_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – EXISTS AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_EXISTS_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_EXISTS_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – OHLC AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_OHLC_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_OHLC_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – PERCENTILE AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_PERCENTILE_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_PERCENTILE_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)

## Experiment – TOPK AGG QUERY
**Perform experiments**

In [ ]:
runWorkload("AGG_TOPK_QUERY", targetThreads, server_local_ips[server_ips[0]], 9497, "XStore", client_ips, 60, 1000, serverGroup, SERVER_WORKING_DIR, "ubuntu")

In [ ]:
unaryPlots("AGG_TOPK_QUERY", "XStore", targetThreads, client_ips, "Aggregate", logScale=True)